# Exploracion y limpieza: categories

Perfilado de Bronze, reglas aplicadas y comparacion lado a lado.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from transformacion.pipeline_transformacion import CLEANERS
from pyspark.sql import functions as F
ENTITY = 'categories'
storage_account = os.getenv('STORAGE_ACCOUNT_NAME', 'stecommercedirty01')
df_raw = spark.read.option('multiLine', True).json(f'abfss://bronze@{storage_account}.dfs.core.windows.net/{ENTITY}/')
print(f'Entidad: {ENTITY} | filas: {df_raw.count():,}')
df_raw.printSchema()

Entidad: categories | filas: 24
root
 |-- created_at: string (nullable = true)
 |-- description: string (nullable = true)
 |-- id: long (nullable = true)
 |-- is_active: string (nullable = true)
 |-- name: string (nullable = true)
 |-- parent_id: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- ingestion_date: date (nullable = true)



In [8]:
df_raw.limit(20).show(30, truncate=False)
df_raw.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df_raw.columns]).show(30, truncate=False)

+----------+-------------------------------+---+---------+----------------+---------+----------+--------------+
|created_at|description                    |id |is_active|name            |parent_id|updated_at|ingestion_date|
+----------+-------------------------------+---+---------+----------------+---------+----------+--------------+
|2022-01-20|Phones, laptops, gadgets       |1  |true     |Electronics     |NULL     |NULL      |2026-08-30    |
|2022-08-17|Shirts, pants, jackets         |2  |true     |Clothing        |NULL     |NULL      |2026-08-30    |
|2022-06-20|Furniture, tools, decor        |3  |true     |Home & Garden   |NULL     |NULL      |2026-08-30    |
|2022-02-27|Equipment, apparel, shoes      |4  |true     |Sports          |NULL     |NULL      |2026-08-30    |
|2030-01-01|Fiction, non-fiction, textbooks|5  |true     |Books           |NULL     |NULL      |2026-08-30    |
|2022-07-28|Kids toys, board games         |6  |true     |Toys            |NULL     |NULL      |2026-08-

In [ ]:
from pyspark.storagelevel import StorageLevel

# Las transformaciones son lazy; persistimos una vez para no recalcular Bronze en cada display.
df_clean = CLEANERS[ENTITY](df_raw, spark).persist(StorageLevel.MEMORY_AND_DISK)
_ = df_clean.count()

pairs = [(c, f'{c}_clean') for c in df_raw.columns if f'{c}_clean' in df_clean.columns]
comparison_columns = []
for raw_name, clean_name in pairs:
    comparison_columns.extend([
        F.col(raw_name).alias(f'{raw_name}_antes'),
        F.col(clean_name).alias(f'{raw_name}_despues'),
    ])

# Una sola accion para inspeccionar todas las columnas, limitada a una muestra.
print('ANTES vs DESPUES:')
df_clean.select(*comparison_columns).limit(100).show(100, truncate=False)
df_clean.printSchema()

In [ ]:
df_clean.select([F.sum((F.col(raw) != F.col(clean)).cast('int')).alias(raw) for raw, clean in pairs]).display() if pairs else None